# Integrasi Data NDVI (Vegetasi) ke Dataset Utama
**Tujuan:** Menambahkan fitur indeks vegetasi (NDVI) ke dataset harian ISPU.

**Metode:**
1.  **Standardisasi:** Mengubah nama kolom dan nama stasiun agar sesuai standar (DKI1-DKI5).
2.  **Resampling:** Mengubah data satelit (~16 harian) menjadi harian menggunakan interpolasi linear berbasis waktu.
3.  **Merging:** Menggabungkan dengan dataset utama dan mengisi *edge cases* (awal/akhir periode) dengan data terdekat (*forward/backward fill*).

In [13]:
import pandas as pd
import numpy as np

# Config
ndvi_path = '../dataset/NDVI (vegetation index)/indeks-ndvi-jakarta.csv'
main_path = 'dataset/merged_data_v3_population.csv'
out_path = 'dataset/merged_data_v4_complete.csv'

# 1. Load Data
df_ndvi = pd.read_csv(ndvi_path)
df_main = pd.read_csv(main_path)

# Normalize Columns & Types
rename_map = {
    'date': 'tanggal', 'system:time_start': 'tanggal',
    'stasiun_id': 'stasiun', 'station': 'stasiun', 'Name': 'stasiun', 'kabko': 'stasiun',
    'average': 'ndvi', 'mean': 'ndvi', 'value': 'ndvi'
}
df_ndvi = df_ndvi.rename(columns=rename_map)
df_ndvi['tanggal'] = pd.to_datetime(df_ndvi['tanggal'])
df_main['tanggal'] = pd.to_datetime(df_main['tanggal'])

# Standardize Station Names
station_map = {
    'JAKARTA PUSAT': 'DKI1', 'DKI1': 'DKI1',
    'JAKARTA UTARA': 'DKI2', 'DKI2': 'DKI2', 
    'JAKARTA SELATAN': 'DKI3', 'DKI3': 'DKI3',
    'JAKARTA TIMUR': 'DKI4', 'DKI4': 'DKI4', 
    'JAKARTA BARAT': 'DKI5', 'DKI5': 'DKI5'
}
if df_ndvi['stasiun'].dtype == object:
    df_ndvi['stasiun'] = df_ndvi['stasiun'].str.upper().str.strip().map(station_map).fillna(df_ndvi['stasiun'])

print(f"Loaded NDVI: {df_ndvi.shape}")

Loaded NDVI: (1810, 3)


In [14]:
# 2. Vectorized Resampling & Interpolation
# Pivot (Date x Station) -> Resample (Daily) -> Interpolate (Time) -> Melt (Long)

# Deduplicate and Pivot
df_pivot = df_ndvi.groupby(['tanggal', 'stasiun'])['ndvi'].mean().reset_index() \
    .pivot(index='tanggal', columns='stasiun', values='ndvi')

# Interpolate
df_daily = df_pivot.resample('D').interpolate(method='time')

# Melt back to long format
df_ndvi_daily = df_daily.reset_index().melt(
    id_vars='tanggal', var_name='stasiun', value_name='ndvi'
)

print(f"Daily NDVI Shape: {df_ndvi_daily.shape}")

Daily NDVI Shape: (28665, 3)


In [15]:
# 3. Merging & Saving
df_final = pd.merge(df_main, df_ndvi_daily, on=['tanggal', 'stasiun'], how='left')

# Fill missing edges (start/end dates)
df_final['ndvi'] = df_final.groupby('stasiun')['ndvi'].transform(lambda x: x.ffill().bfill())

# Save
df_final.to_csv(out_path, index=False)

print(f"Saved to: {out_path}")
print(f"Final Info: {len(df_final)} rows, {df_final['ndvi'].isna().sum()} missing NDVI")

Saved to: dataset/merged_data_v4_complete.csv
Final Info: 15412 rows, 2 missing NDVI
